In [21]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

In [22]:
# Clone or update our BMEN 600 group's GitHub repository.
repo_path = Path("/content/BMEN-600-Group-19")

if repo_path.exists():
    !git -C /content/BMEN-600-Group-19 pull origin main
else:
    !git clone https://github.com/chelseataylor1/BMEN-600-Group-19.git /content/BMEN-600-Group-19

From https://github.com/chelseataylor1/BMEN-600-Group-19
 * branch            main       -> FETCH_HEAD
Already up to date.


In [23]:
# Locate the output dataset from Notebook 2.
zip_path = repo_path / "data/processed/OutputFromNotebook2_EHG_Dataset.zip"

# Extract the dataset.
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall("/content")

# Define the locations of the extracted files.
dataset_path = Path("/content/BMEN600_Matched_Trimmed_EHG_Dataset")
recordings_path = dataset_path / "recordings"
metadata_path = dataset_path / "matched_pregnancies.csv"

print("Dataset extracted successfully!")
print("Recordings:", recordings_path)
print("Metadata:", metadata_path)

Dataset extracted successfully!
Recordings: /content/BMEN600_Matched_Trimmed_EHG_Dataset/recordings
Metadata: /content/BMEN600_Matched_Trimmed_EHG_Dataset/matched_pregnancies.csv


In [24]:
# Load the Matched Pregnancy Metadata.
df_matched = pd.read_csv(metadata_path)

# Confirm the metadata loaded successfully.
print("Matched pregnancies loaded:", len(df_matched))

# Preview the metadata.
display(df_matched.head())

Matched pregnancies loaded: 35


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1


In [25]:
# Verify the trimmed EHG recordings.

# Find all trimmed EHG recording CSV files.
recording_files = sorted(recordings_path.glob("*.csv"))

# Confirm the number of recordings.
print("Trimmed EHG recordings loaded:", len(recording_files))

# Preview the contents of one recording.
if recording_files:
    recording = pd.read_csv(recording_files[0])

    print("Recording:", recording_files[0].stem)
    print("Available variables:", recording.columns.tolist())
    print("Number of samples:", len(recording))

    display(recording.head())
else:
    print("No CSV recordings found. Check the dataset path.")

Trimmed EHG recordings loaded: 70
Recording: icehg1010
Available variables: ['trimmed_time', 'trimmed_S1', 'trimmed_S2', 'trimmed_S3', 'trimmed_S4']
Number of samples: 29199


,trimmed_time,trimmed_S1,trimmed_S2,trimmed_S3,trimmed_S4
0,2.500833,0.007172,0.021744,-0.000076,-0.014649
1,2.501667,0.007172,0.021973,-0.001678,-0.016480
2,2.502500,0.006027,0.020600,-0.002518,-0.017090
3,2.503333,0.001221,0.015412,0.000229,-0.013962
4,2.504167,-0.004349,0.008927,0.004730,-0.008545


In [26]:
"""
Median frequency (MF) of EHG channels S1-S4 for every matched pregnancy
with outcome = "induced-cesarean".

For each pregnancy the early (~23 wk) and later (~31 wk) recordings are
loaded, and MF is computed for S1, S2, S3 and S4. Output: one table with a
row per pregnancy (early MF, later MF and the change later - early).

Method: each trimmed, filtered recording is converted to a power spectrum
(whole recording, no windows), and the median frequency is the frequency that
splits the power between 0.3 and 3 Hz into two equal halves. The signals are
used as provided (no extra filtering here).

Change columns:
  *_change          = later MF - early MF (Hz)
  *_change_per_week = *_change / weeks_between (Hz per week), where
                      weeks_between = later_week - early_week for that pair

Packages: numpy, pandas, scipy
"""

import os
import numpy as np
import pandas as pd
from scipy.signal import periodogram

# ----------------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------------
OUTCOME = "induced-cesarean"            # induced | cesarean | induced-cesarean
CHANNELS = ["S1", "S2", "S3", "S4"]
BAND = (0.3, 3.0)      # Hz, band in which the median frequency is calculated
OUT_CSV = f"median_frequency_{OUTCOME}.csv"


def power_spectrum(x, fs):
    """Power spectrum of the whole trimmed, filtered recording (one FFT-based
    periodogram, no windowing into segments; Hann taper, mean removed)."""
    x = np.asarray(x, dtype=float)
    return periodogram(x, fs=fs, window="hann", detrend="constant",
                       scaling="density")


def median_frequency(x, fs, band=BAND):
    """Frequency (Hz) that splits the power spectrum within `band` into two
    halves of equal power (linear interpolation of the cumulative power)."""
    freqs, psd = power_spectrum(x, fs)

    keep = (freqs >= band[0]) & (freqs <= band[1])
    freqs, psd = freqs[keep], psd[keep]

    cum = np.cumsum(psd)
    cum = cum / cum[-1]
    return float(np.interp(0.5, cum, freqs))


def load_recording(rec_id):
    # Locate the CSV recording.
    path = recordings_path / f"icehg{int(rec_id)}.csv"


    # Load the processed EHG recording.
    data = pd.read_csv(path)

    # Retrieve the sampling frequency.
    if "fs" in data.columns:
        fs = float(data["fs"].iloc[0])
    else:
        # Calculate sampling frequency from time in minutes.
        fs = 1 / (np.median(np.diff(data["trimmed_time"])) * 60)

    # Extract the four EHG channels as NumPy arrays.
    channels = {
        "S1": data["trimmed_S1"].to_numpy(),
        "S2": data["trimmed_S2"].to_numpy(),
        "S3": data["trimmed_S3"].to_numpy(),
        "S4": data["trimmed_S4"].to_numpy()
    }

    return channels, fs


def mf_for_recording(rec_id):
    channels, fs = load_recording(rec_id)
    return {ch: median_frequency(sig, fs) for ch, sig in channels.items()}

# Calculating the Change in Median Frequency per Weekly Recordings: 23 & 31 Weeks (Jerold)

**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across ALL pregnancies**

In [27]:
print(df_matched.columns.tolist())

['pregnancy_id', 'age', 'early_id', 'early_week', 'weight_early', 'later_id', 'later_week', 'weight_later', 'height', 'placenta', 'gestation', 'newborn_weight', 'outcome', 'weeks_between']


In [28]:
# 1. Filter for the target outcome group (cesarean) and exclude recording ID 1393
df_group = df_matched[df_matched["outcome"] == "induced-cesarean"].copy()
df_group = df_group[(df_group["early_id"] != 1393) & (df_group["later_id"] != 1393)]

individual_results = []

for _, row in df_group.iterrows():
    preg_id = int(row["pregnancy_id"])
    weeks_between = float(row["weeks_between"])

    # Load and calculate early (~23 wk) and later (~31 wk) MF for channels S1-S4
    early_mf = mf_for_recording(row["early_id"])
    later_mf = mf_for_recording(row["later_id"])

    preg_data = {
        "pregnancy_id": preg_id,
        "early_week": row["early_week"],
        "later_week": row["later_week"],
        "weeks_between": weeks_between,
    }

    # Compute absolute change and weekly rate per channel
    for ch in CHANNELS:
        change = later_mf[ch] - early_mf[ch]
        preg_data[f"{ch}_early_MF"] = round(early_mf[ch], 4)
        preg_data[f"{ch}_later_MF"] = round(later_mf[ch], 4)
        preg_data[f"{ch}_change"] = round(change, 4)
        preg_data[f"{ch}_change_per_week"] = round(
            change / weeks_between if weeks_between > 0 else 0.0, 4
        )

    individual_results.append(preg_data)

# Define table using the calculated DataFrame
df_induced_cesarean_mf = pd.DataFrame(individual_results)
table = df_induced_cesarean_mf

# Extract value columns for calculation
mf_cols = ['S1_early_MF', 'S1_later_MF', 'S1_change', 'S1_change_per_week',
           'S2_early_MF', 'S2_later_MF', 'S2_change', 'S2_change_per_week',
           'S3_early_MF', 'S3_later_MF', 'S3_change', 'S3_change_per_week',
           'S4_early_MF', 'S4_later_MF', 'S4_change', 'S4_change_per_week']

# Generate summary table for mean, median, and std
summary_stats = table[mf_cols].agg(['mean', 'median', 'std']).round(4)
print(summary_stats.T)

                      mean  median     std
S1_early_MF         0.5337  0.3984  0.3054
S1_later_MF         0.6233  0.5129  0.2589
S1_change           0.0896  0.1187  0.4523
S1_change_per_week  0.0108  0.0108  0.0569
S2_early_MF         0.4161  0.4175  0.0570
S2_later_MF         0.3967  0.3758  0.0427
S2_change          -0.0194 -0.0200  0.0599
S2_change_per_week -0.0026 -0.0027  0.0073
S3_early_MF         0.4300  0.4103  0.0679
S3_later_MF         0.4610  0.4080  0.0758
S3_change           0.0309  0.0138  0.0877
S3_change_per_week  0.0039  0.0013  0.0108
S4_early_MF         0.4014  0.3947  0.0449
S4_later_MF         0.4970  0.4723  0.1233
S4_change           0.0955  0.0853  0.1473
S4_change_per_week  0.0114  0.0109  0.0179


**Change in Median Frequency between early (23 weeks) & later (31 weeks) EHG recordings across individual pregnancies**

In [29]:
# 1. Filter for the target outcome group (cesarean) and exclude recording ID 1393
df_group = df_matched[df_matched["outcome"] == "induced-cesarean"].copy()
df_group = df_group[(df_group["early_id"] != 1393) & (df_group["later_id"] != 1393)]

individual_results = []

for _, row in df_group.iterrows():
    preg_id = int(row["pregnancy_id"])
    weeks_between = float(row["weeks_between"])

    # Load and calculate early (~23 wk) and later (~31 wk) MF for channels S1-S4
    early_mf = mf_for_recording(row["early_id"])
    later_mf = mf_for_recording(row["later_id"])

    preg_data = {
        "pregnancy_id": preg_id,
        "early_week": row["early_week"],
        "later_week": row["later_week"],
        "weeks_between": weeks_between,
    }

    # Compute absolute change and weekly rate per channel
    for ch in CHANNELS:
        change = later_mf[ch] - early_mf[ch]
        preg_data[f"{ch}_early_MF"] = round(early_mf[ch], 4)
        preg_data[f"{ch}_later_MF"] = round(later_mf[ch], 4)
        preg_data[f"{ch}_change"] = round(change, 4)
        preg_data[f"{ch}_change_per_week"] = round(
            change / weeks_between if weeks_between > 0 else 0.0, 4
        )

    individual_results.append(preg_data)

# 2. Convert to DataFrame and display individual results
df_induced_cesarean_mf = pd.DataFrame(individual_results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
display(df_induced_cesarean_mf)

,pregnancy_id,early_week,later_week,weeks_between,S1_early_MF,S1_later_MF,S1_change,S1_change_per_week,S2_early_MF,S2_later_MF,S2_change,S2_change_per_week,S3_early_MF,S3_later_MF,S3_change,S3_change_per_week,S4_early_MF,S4_later_MF,S4_change,S4_change_per_week
0,79,22.9,30.3,7.4,0.3961,0.7195,0.3235,0.0437,0.3808,0.3608,-0.0200,-0.0027,0.4103,0.4002,-0.0101,-0.0014,0.3947,0.4723,0.0776,0.0105
1,80,22.4,30.6,8.2,0.3984,0.6893,0.2909,0.0355,0.4236,0.4616,0.0380,0.0046,0.4318,0.5622,0.1304,0.0159,0.3965,0.5186,0.1221,0.0149
2,81,22.9,33.9,11.0,0.3943,0.5129,0.1187,0.0108,0.3962,0.4333,0.0371,0.0034,0.3942,0.4080,0.0138,0.0013,0.3942,0.5273,0.1331,0.0121
3,82,22.4,31.1,8.7,0.5331,0.4605,-0.0726,-0.0083,0.4175,0.3723,-0.0452,-0.0052,0.4851,0.3889,-0.0962,-0.0111,0.4181,0.3429,-0.0751,-0.0086
4,83,22.9,30.7,7.8,0.3629,0.3889,0.0260,0.0033,0.3357,0.3758,0.0401,0.0051,0.3650,0.5222,0.1572,0.0201,0.3515,0.4368,0.0853,0.0109
5,84,23.0,30.9,7.9,1.2151,0.4531,-0.7620,-0.0965,0.5202,0.4241,-0.0960,-0.0122,0.5533,0.5380,-0.0153,-0.0019,0.4900,0.4415,-0.0486,-0.0061
6,85,23.6,31.7,8.1,0.4362,1.1389,0.7027,0.0868,0.4387,0.3487,-0.0899,-0.0111,0.3704,0.4072,0.0368,0.0045,0.3649,0.7393,0.3744,0.0462


**Analysis of median frequency change**

1. S1 and S4 exhibit clear, positive increases in median frequency (0.1187 & 0.0853 Hz)

In [30]:
import shutil
from pathlib import Path
from google.colab import files

# 1. Copy the original dataset into a new folder
output_folder = Path("/content/BMEN600_MF_Dataset")
shutil.copytree(dataset_path, output_folder, dirs_exist_ok=True)

# 2. Add the new median frequency variables to the original metadata
df_updated = df_matched.merge(
    df_induced_cesarean_mf,
    on="pregnancy_id",
    how="left"
)

# 3. Save the updated CSV
df_updated.to_csv(output_folder / "matched_pregnancies.csv", index=False)

# 4. Create the ZIP file
zip_path = shutil.make_archive(
    "/content/BMEN600_MF_Dataset",
    "zip",
    root_dir="/content",
    base_dir="BMEN600_MF_Dataset"
)

# 5. Download the ZIP
print("ZIP created successfully!")
files.download(zip_path)

ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>